
# Structured Streaming

## In this lesson you:
* Learn about Structured Streaming at a high level

## What is Structured Streaming?
A stream is a sequence of data that is made available over time. Structured Streaming where we treat a <b>stream</b> of data as a table to which data is continously appended. 

The developer then defines a query on this input table, as if it were a static table, to compute a final result table that will be written to an output <b>sink</b>.


## The Problem

We have a stream of data coming in from a TCP-IP socket, Kafka, Kinesis or other sources...

The data is coming in faster than it can be consumed

How do we solve this problem?
<br />


<img src="https://files.training.databricks.com/images/drinking-from-the-fire-hose.png"/>


<h2> The Micro-Batch Model</h2>

Many APIs solve this problem by employing a Micro-Batch model.

In this model, we take our firehose of data and collect data for a set interval of time (the **Trigger Interval**).

In our example, the **Trigger Interval** is two seconds.

<img style="width:100%" src="https://files.training.databricks.com/images/streaming-timeline.png"/>


<h2> Processing the Micro-Batch</h2>

For each interval, our job is to process the data from the previous [two-second] interval.

As we are processing data, the next batch of data is being collected for us.

In our example, we are processing two seconds worth of data in about one second.

<img style="width:100%" src="https://files.training.databricks.com/images/streaming-timeline-1-sec.png">

## What happens if we don't process the data fast enough when reading from a TCP/IP Stream?

In the case of a TCP/IP stream, we will most likely drop packets.
In other words, we would be losing data.
If this is an IoT device measuring the outside temperature every 15 seconds, this might be OK.
If this is a critical shift in stock prices, you could be losing out thousands of dollars.

## What happens if we don't process the data fast enough when reading from a pubsub system like Kafka?

In the case of a pubsub system, it simply means we fall further behind.
Eventually, the pubsub system would reach resource limits inducing other problems.
However, we can always re-launch / resend the cluster with enough cores to catch up and stay current.

Our goal is simply to process the data for the previous interval before data from the next interval arrives.


<h2> From Micro-Batch to Table</h2>

In Apache Spark, we treat such a stream of **micro-batches** as continuous updates to a table.

The developer then defines a query on this **input table**, as if it were a static table.

The computation on the input table is then pushed to a **results table**.

And finally, the results table is written to an output **sink**. 

<img src="https://files.training.databricks.com/images/eLearning/Delta/stream2rows.png" style="height: 300px"/>

In general, Spark Structured Streams consist of two parts:
* The **Input source** such as 
  * Kafka
  * Azure Event Hub
  * Files on a distributed system
  * TCP-IP sockets
* And the **Sinks** such as
  * Kafka
  * Azure Event Hub
  * Various file formats
  * The system console
  * Apache Spark tables (memory sinks)
  * The completely custom `foreach()` iterator

### Update Triggers
Developers define **triggers** to control how frequently the **input table** is updated. 

Each time a trigger fires, Spark checks for new data (new rows for the input table), and updates the result.

From the docs for `DataStreamWriter.trigger(Trigger)`:
> The default value is ProcessingTime(0) and it will run the query as fast as possible.

And the process repeats in perpetuity.


<h2> More About Streams</h2>

Examples include bank card transactions, log files, Internet of Things (IoT) device data, video game play events and countless others.

Some key properties of streaming data include:
* Data coming from a stream is typically not ordered in any way
* The data is streamed into a **data lake**
* The data is coming in faster than it can be consumed
* Streams are often chained together to form a data pipeline
* Streams don't have to run 24/7:
  * Consider the new log files that are processed once an hour
  * Or the financial statement that is processed once a month